# Phase 2 — Interactive Review (Section 2.3)

Run `phase2_generate.py` from the terminal first (data prep + dual generation). This notebook picks up where that script leaves off -- loading the saved checkpoints and doing the parts that genuinely need a Jupyter widget: selecting concepts and reading review sheets.

Uses the same venv as the script -- make sure this notebook's kernel is set to that `.venv` (kernel picker, top-right).

## Setup: reload everything the script produced

In [1]:
import os
import pickle
import json
import itertools
import pandas as pd
from dotenv import load_dotenv

import text_lloom.workbench as wb
from text_lloom.llm import Model, EmbedModel

pd.set_option('display.max_colwidth', 200)

OUTPUT_DIR = "/Users/nadia/Desktop/redditRun_june/comment_data/"
ENV_PATH = os.path.join(OUTPUT_DIR, ".env")
CKPT_DIR = os.path.join(OUTPUT_DIR, "ckpt")

TEXT_COL = "body"
ID_COL = "id"
SUBREDDIT_COL = "subreddit_source"

loaded = load_dotenv(ENV_PATH)
print(f".env loaded: {loaded}")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
print(f"Key set: {bool(OPENROUTER_API_KEY)}, length: {len(OPENROUTER_API_KEY) if OPENROUTER_API_KEY else 0}")

OPENROUTER_MODEL = "openai/gpt-5.6-luna-pro"   # match whatever phase2_generate.py used
MODEL_COST = (0.20 / 1_000_000, 1.20 / 1_000_000)
CONTEXT_WINDOW = 1_050_000
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>


.env loaded: True
Key set: True, length: 73


In [2]:
def setup_llm_fn(api_key):
    from openai import AsyncOpenAI
    import httpx
    return AsyncOpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
        timeout=httpx.Timeout(connect=15.0, read=120.0, write=120.0, pool=120.0),
    )

def setup_embed_fn(api_key):
    from fastembed import TextEmbedding
    return TextEmbedding(model_name=EMBED_MODEL_NAME)

import asyncio, random

MAX_RETRIES = 5
BASE_DELAY = 5
MAX_OUTPUT_TOKENS = 4096

async def call_llm_fn(model, prompt):
    if "system_prompt" not in model.args:
        model.args["system_prompt"] = (
            "You are a helpful assistant who helps with identifying patterns in text examples."
        )
    if "temperature" not in model.args:
        model.args["temperature"] = 0

    for attempt in range(MAX_RETRIES):
        try:
            res = await model.client.chat.completions.create(
                model=model.name,
                temperature=model.args["temperature"],
                max_tokens=MAX_OUTPUT_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": model.args["system_prompt"]},
                    {"role": "user", "content": prompt},
                ],
            )
            text = res.choices[0].message.content if res and res.choices else None
            tokens = (res.usage.prompt_tokens, res.usage.completion_tokens) if res and getattr(res, "usage", None) else (0, 0)
            return text, tokens
        except Exception as e:
            err_str = str(e).lower()
            is_credits = "402" in str(e) or "requires more credits" in err_str
            is_last_attempt = attempt == MAX_RETRIES - 1
            is_retryable = (
                "429" in str(e) or "rate limit" in err_str or "timed out" in err_str
                or "timeout" in err_str or "connection" in err_str
            )
            if is_credits:
                print(f"  [402 -- out of credits] {e}")
                return None, None
            if is_retryable and not is_last_attempt:
                delay = BASE_DELAY * (2 ** attempt) + random.uniform(0, 2)
                print(f"  [{type(e).__name__}] retrying in {delay:.1f}s (attempt {attempt + 1}/{MAX_RETRIES})...")
                await asyncio.sleep(delay)
                continue
            print(f"  [error, giving up after {attempt + 1} attempt(s)]: {e}")
            return None, None
    return None, None

def call_embed_fn(model, text_arr):
    embeddings = [e.tolist() for e in model.client.embed(text_arr)]
    return embeddings, (0, 0)

def build_models():
    return dict(
        distill_model=Model(setup_fn=setup_llm_fn, fn=call_llm_fn, name=OPENROUTER_MODEL,
                             cost=MODEL_COST, rate_limit=(15, 10), context_window=CONTEXT_WINDOW, api_key=OPENROUTER_API_KEY),
        cluster_model=EmbedModel(setup_fn=setup_embed_fn, fn=call_embed_fn, name=EMBED_MODEL_NAME,
                                  cost=0, batch_size=64, api_key=OPENROUTER_API_KEY),
        synth_model=Model(setup_fn=setup_llm_fn, fn=call_llm_fn, name=OPENROUTER_MODEL,
                           cost=MODEL_COST, rate_limit=(10, 10), context_window=CONTEXT_WINDOW, api_key=OPENROUTER_API_KEY),
        score_model=Model(setup_fn=setup_llm_fn, fn=call_llm_fn, name=OPENROUTER_MODEL,
                           cost=MODEL_COST, rate_limit=(10, 10), context_window=CONTEXT_WINDOW, api_key=OPENROUTER_API_KEY),
    )

## Load the sample and the seeded/unseeded checkpoints from the script

In [3]:
gen_sample = pd.read_parquet(os.path.join(OUTPUT_DIR, "gen_sample.parquet"))
print(f"gen_sample: {len(gen_sample):,} rows")

def load_checkpoint(name):
    path = os.path.join(CKPT_DIR, f"{name}.pkl")
    assert os.path.exists(path), f"Checkpoint not found: {path} -- run phase2_generate.py first."
    with open(path, "rb") as f:
        l = pickle.load(f)
    for k, v in build_models().items():
        setattr(l, k, v)
    return l

l_unseeded = load_checkpoint("final_unseeded")
l_seeded = load_checkpoint("final_seeded")
print(f"Unseeded: {len(l_unseeded.concepts)} concepts")
print(f"Seeded: {len(l_seeded.concepts)} concepts")

gen_sample: 2,036 rows
Unseeded: 4 concepts
Seeded: 9 concepts


---
## Section 2.3 -- pick a list, select, score, review

Iterative from here -- not top-to-bottom.

In [4]:
# Pick which lloom instance to review -- change if reconciling both lists.
l = l_seeded

**Run this cell, then STOP and actually click checkboxes in the widget before running anything else.** `l.select()` renders the UI but doesn't pause execution to wait for you.

In [5]:
l.select()

### Diagnostic -- run before scoring

In [6]:
import json as _json
widget_state = _json.loads(l.select_widget.data)
n_active = sum(1 for v in widget_state.values() if v.get("active"))
print(f"Total concepts: {len(widget_state)}, active: {n_active}")
for c_id, info in widget_state.items():
    print(f"  {c_id}: active={info.get('active')}  name={info.get('name')}")
if n_active == 0:
    print("\n⚠️  Nothing selected yet -- go back and click checkboxes in the widget above.")

Total concepts: 9, active: 6
  9db03388-7d8a-495d-94af-817a976a7bac: active=True  name=Workplace Problem Guidance
  04330b54-bb09-4973-b644-38076e61f2ba: active=False  name=Career Planning Advice
  1e6986fb-b8ee-43be-a587-9c7eb6e945c6: active=True  name=Emotional Support
  7878b017-ead7-45fc-bf1b-457cc8639e8f: active=True  name=Critical Pushback
  18bf112a-a404-46b8-bb73-db1b539e8367: active=True  name=Personal Relating
  9caa3b4a-2479-44d9-8598-6d21493c5ff1: active=False  name=Discussion Direction
  a3b74b1c-4404-4be4-bf8d-60bbb7b0e231: active=True  name=Emotional Encouragement
  1ce29757-8c2b-405c-aa44-1fb17f874d9f: active=True  name=Practical Advice
  adb41517-0ed6-474d-9376-20985982a20c: active=False  name=Support Connections


### Score -- chunked + checkpointed

Scoring the full `gen_sample` in one call was hitting `APITimeoutError` under real-size prompts (5 comments + criteria bundled per request, expecting a longer structured JSON response) even though smaller test batches worked fine. Chunked here instead: each `CHUNK_SIZE`-row slice gets its own `l.score()` call and its own saved file, written immediately -- so a failure partway through only costs the chunk in progress, and re-running this cell skips whatever's already scored rather than starting over.

In [7]:
import time
import math
import glob

CHUNK_SIZE = 200       # confirmed clean at this size; drop lower if this still times out
SCORE_BATCH_SIZE = 5   # confirmed fine at this size

SCORE_CHUNK_DIR = os.path.join(OUTPUT_DIR, "score_chunks")
os.makedirs(SCORE_CHUNK_DIR, exist_ok=True)

n_chunks = math.ceil(len(gen_sample) / CHUNK_SIZE)
print(f"gen_sample: {len(gen_sample):,} rows -> {n_chunks} chunks of up to {CHUNK_SIZE} rows each")

for chunk_idx in range(n_chunks):
    chunk_path = os.path.join(SCORE_CHUNK_DIR, f"score_chunk_{chunk_idx:03d}.parquet")
    if os.path.exists(chunk_path):
        print(f"[{chunk_idx+1}/{n_chunks}] already scored -- skipping")
        continue

    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(gen_sample))
    chunk_df = gen_sample.iloc[start:end]

    t0 = time.time()
    print(f"[{chunk_idx+1}/{n_chunks}] scoring rows {start}:{end} (n={len(chunk_df)})...")
    chunk_sc = await l.score(get_highlights=True, debug=False, batch_size=SCORE_BATCH_SIZE,
                              ignore_existing=False, df=chunk_df)
    chunk_sc.to_parquet(chunk_path)
    elapsed = time.time() - t0
    print(f"  done in {elapsed:.1f}s -- saved {os.path.basename(chunk_path)}")

    if chunk_idx == 0:
        est_total_min = (elapsed * n_chunks) / 60
        print(f"  --> Based on this first chunk, full run estimated at ~{est_total_min:.1f} minutes total.")

print("\nAll chunks present. Combining...")
chunk_files = sorted(glob.glob(os.path.join(SCORE_CHUNK_DIR, "score_chunk_*.parquet")))
sc = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)
sc.to_parquet(os.path.join(OUTPUT_DIR, "gen_scores.parquet"))
print(f"Combined: {len(sc):,} scored rows -> saved to gen_scores.parquet")

likely_failed = sc[(sc["score"] == 0) & (sc.get("rationale", "") == "") & (sc.get("highlight", "") == "")]
print(f"Likely silent failures: {len(likely_failed):,} / {len(sc):,} ({len(likely_failed)/len(sc):.1%})")
if len(likely_failed) / len(sc) > 0.05:
    print("⚠️  Over 5% -- some chunks may have hit the same timeout issue. Check which chunk files "
          "correspond to the affected rows and consider deleting + re-running just those.")

gen_sample: 2,036 rows -> 11 chunks of up to 200 rows each
[1/11] scoring rows 0:200 (n=200)...
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 6/6 [01:15<00:00, 12.62s/it]
✅ Done with concept scoring!
  done in 75.8s -- saved score_chunk_000.parquet
  --> Based on this first chunk, full run estimated at ~13.9 minutes total.
[2/11] scoring rows 200:400 (n=200)...
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 6/6 [01:21<00:00, 13.65s/it]
✅ Done with concept scoring!
  done in 81.9s -- saved score_chunk_001.parquet
[3/11] scoring rows 400:600 (n=200)...
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 6/6 [01:15<00:00, 12.64s/it]
✅ Done with concept scoring!
  done in 75.9s -- saved score_chunk_002.parquet
[4/11] scoring rows 600:800 (n=200)...
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 6/6 [01:20<00:00, 13.50s/it]
✅ Done wi

### Review sheet per concept

In [8]:
for cid, grp in sc[sc.score > 0].groupby("concept_id"):
    name = grp.concept_name.iloc[0]
    prompt = grp.concept_prompt.iloc[0]
    n = len(grp)
    print(f"\n{'='*70}\nCONCEPT {cid}: {name}")
    print(f"CRITERIA: {prompt}")
    print(f"PREVALENCE: {n} / {gen_sample.shape[0]} = {n/gen_sample.shape[0]:.1%}\n")
    for t in grp.text.sample(min(20, n), random_state=0):
        print(f"  - {t[:250]}")


CONCEPT 18bf112a-a404-46b8-bb73-db1b539e8367: Personal Relating
CRITERIA: Determine whether the commenter connects with the post author by sharing a similar personal, professional, or emotional experience or perspective.
PREVALENCE: 1745 / 2036 = 85.7%

  - Hounding, pressure, blame - at some point you have to disassociate from all this. You are not your job, and you really shouldnt stress about your job if the bills are being paid nicely and you are doing enough to be considered adequately performing y
  - A close friend moved into a research role after many years of incident response and is much happier
  - I will follow this topic, all this technical is start to become a thankless job.
  - AppSec is really hot right now, every other companies is hiring AppSec person. It’s the other side of VA/Pentest, ensuring the application or “product” delivered by developers are not riddled with vulnerabilities. The work is slower compared to Pente
  - Frankly, your situation sounds bad, but yo

### Four questions per concept (spreadsheet, done by hand)

| Question | If no |
|---|---|
| Do the 20 examples share the thing the name claims? | Rewrite criteria, or drop |
| Is the name an action rather than a topic? | Rewrite name and criteria |
| Is it distinct from every other concept? | Merge |
| Is prevalence between ~2% and 50%? | Under 2%: drop. Over 50%: split or drop |


### Mechanical redundancy check (Jaccard)

In [9]:
w = sc.pivot(index="doc_id", columns="concept_name", values="score").fillna(0)
for a, b in itertools.combinations(w.columns, 2):
    inter = ((w[a] == 1) & (w[b] == 1)).sum()
    union = ((w[a] == 1) | (w[b] == 1)).sum()
    j = inter / union if union else 0
    if j > 0.5:
        print(f"MERGE CANDIDATE  {j:.2f}  {a}  ||  {b}")

MERGE CANDIDATE  0.68  Emotional Encouragement  ||  Emotional Support


### Merge redundant concepts

Two concepts fired on 68% of the same comments (Jaccard 0.68 -- well above the 0.5 merge threshold). 


In [10]:
KEEP_NAME = "Emotional Support"       # concept to keep -- edit if you'd rather keep the other one
DROP_NAME = "Emotional Encouragement"  # concept being retired into KEEP_NAME

def find_concept_by_name(name):
    matches = [(c_id, c) for c_id, c in l.concepts.items() if c.name == name]
    assert len(matches) == 1, f"Expected exactly 1 concept named {name!r}, found {len(matches)}"
    return matches[0]

keep_id, keep_concept = find_concept_by_name(KEEP_NAME)
drop_id, drop_concept = find_concept_by_name(DROP_NAME)

print(f"KEEP [{keep_id}] {keep_concept.name}")
print(f"  {keep_concept.prompt}\n")
print(f"DROP [{drop_id}] {drop_concept.name}")
print(f"  {drop_concept.prompt}")

KEEP [1e6986fb-b8ee-43be-a587-9c7eb6e945c6] Emotional Support
  Determine whether the commenter validates, reassures, empathizes with, or encourages the post author and their emotional well-being.

DROP [a3b74b1c-4404-4be4-bf8d-60bbb7b0e231] Emotional Encouragement
  Determine whether the commenter empathizes with, validates, reassures, encourages, or celebrates the post author in response to their experiences or emotions.


**Write the merged name + criteria below** -- read both prompts printed above and combine them into one description that covers what both were actually catching. Don't just concatenate the two prompts; write it the way you'd write any single concept's criteria.

In [11]:
NEW_NAME = "Emotional Support"  #  EDIT ME
NEW_PROMPT = (

   "Determine whether the commenter validates, reassures, empathizes with, or ,"
    "encourages the post author and their emotional well-being. Includes validates,"
    "reassures, encourages, or celebrates the post author in response to their experiences "
    "or emotions."
)  #  EDIT ME -- draft only, combine the two printed above in your own words

keep_concept.name = NEW_NAME
keep_concept.prompt = NEW_PROMPT
print(f"Updated [{keep_id}]: {keep_concept.name}\n  {keep_concept.prompt}")
print(f"\n{drop_id} ({drop_concept.name}) will be excluded at the freeze step below -- "
      f"not deactivated here, since l.score() re-syncs .active from the widget and would "
      f"silently undo a manual change.")

Updated [1e6986fb-b8ee-43be-a587-9c7eb6e945c6]: Emotional Support
  Determine whether the commenter validates, reassures, empathizes with, or ,encourages the post author and their emotional well-being. Includes validates,reassures, encourages, or celebrates the post author in response to their experiences or emotions.

a3b74b1c-4404-4be4-bf8d-60bbb7b0e231 (Emotional Encouragement) will be excluded at the freeze step below -- not deactivated here, since l.score() re-syncs .active from the widget and would silently undo a manual change.


### Rescore just the merged concept

`c_ids=[keep_id]` restricts scoring to only this one concept -- everything else in `sc` stays as-is, no need to redo work that didn't change. Chunked the same way as the original scoring pass, since this hits the same real-prompt-size timeout risk.

In [12]:
# Confirm keep_id is still active in the widget's synced state -- score()'s
# c_ids filtering still requires this, even though we're targeting one concept.
widget_state_check = _json.loads(l.select_widget.data)
assert widget_state_check[keep_id]["active"], (
    f"{keep_id} isn't marked active in the widget state -- re-run l.select() and "
    f"make sure it's checked before rescoring."
)

MERGE_CHUNK_DIR = os.path.join(OUTPUT_DIR, "score_chunks_merge")
os.makedirs(MERGE_CHUNK_DIR, exist_ok=True)

n_chunks = math.ceil(len(gen_sample) / CHUNK_SIZE)
print(f"Rescoring {KEEP_NAME!r} across {n_chunks} chunks...")

for chunk_idx in range(n_chunks):
    chunk_path = os.path.join(MERGE_CHUNK_DIR, f"merge_chunk_{chunk_idx:03d}.parquet")
    if os.path.exists(chunk_path):
        print(f"[{chunk_idx+1}/{n_chunks}] already scored -- skipping")
        continue

    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(gen_sample))
    chunk_df = gen_sample.iloc[start:end]

    t0 = time.time()
    chunk_sc = await l.score(c_ids=[keep_id], get_highlights=True, debug=False,
                              batch_size=SCORE_BATCH_SIZE, ignore_existing=False, df=chunk_df)
    chunk_sc.to_parquet(chunk_path)
    print(f"[{chunk_idx+1}/{n_chunks}] done in {time.time()-t0:.1f}s")

merge_chunk_files = sorted(glob.glob(os.path.join(MERGE_CHUNK_DIR, "merge_chunk_*.parquet")))
merged_sc = pd.concat([pd.read_parquet(f) for f in merge_chunk_files], ignore_index=True)
print(f"\nRescored: {len(merged_sc):,} rows for {KEEP_NAME!r}")

Rescoring 'Emotional Support' across 11 chunks...
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 1/1 [00:33<00:00, 33.24s/it]
✅ Done with concept scoring!
[1/11] done in 33.3s
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 1/1 [00:26<00:00, 26.82s/it]
✅ Done with concept scoring!
[2/11] done in 26.8s
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 1/1 [00:28<00:00, 28.96s/it]
✅ Done with concept scoring!
[3/11] done in 29.0s
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 1/1 [00:31<00:00, 31.54s/it]
✅ Done with concept scoring!
[4/11] done in 31.5s
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 1/1 [00:31<00:00, 31.50s/it]
✅ Done with concept scoring!
[5/11] done in 31.5s
Cost estimates not available for score model `openai/gpt-5.6-luna-pro`
100%|██████████| 1/1 [00:31<00:00, 31.28s/it

### Update `sc`: drop both old entries, add the merged one back in

Removes stale rows for the old `keep_id` criteria (now outdated) and all rows for `drop_id` (retired), then adds the freshly-scored merged concept back in.

In [13]:
before = len(sc)
sc = sc[~sc["concept_id"].isin([keep_id, drop_id])]
sc = pd.concat([sc, merged_sc], ignore_index=True)
print(f"sc: {before:,} -> {len(sc):,} rows "
      f"({sc['concept_id'].nunique()} concepts remaining, was tracking {before and sc['concept_id'].nunique()+1})")

sc.to_parquet(os.path.join(OUTPUT_DIR, "gen_scores.parquet"))
print("Saved updated gen_scores.parquet")

sc: 12,216 -> 10,180 rows (5 concepts remaining, was tracking 6)
Saved updated gen_scores.parquet


## Rerun the Jaccard check-- to confirm the overlap is resolved

In [14]:
w = sc.pivot(index="doc_id", columns="concept_name", values="score").fillna(0)
found_any = False
for a, b in itertools.combinations(w.columns, 2):
    inter = ((w[a] == 1) & (w[b] == 1)).sum()
    union = ((w[a] == 1) | (w[b] == 1)).sum()
    j = inter / union if union else 0
    if j > 0.5:
        print(f"MERGE CANDIDATE  {j:.2f}  {a}  ||  {b}")
        found_any = True
if not found_any:
    print(" No remaining pairs above 0.5 Jaccard.")

 No remaining pairs above 0.5 Jaccard.


Track the retired concept ids for the step step below

In [15]:
RETIRED_CONCEPT_IDS = [drop_id]  # append to this list if you merge/drop other concepts later

### Freeze to JSON (target 12-16 concepts)

In [16]:
# Excludes RETIRED_CONCEPT_IDS explicitly, rather than relying on c.active --
# l.score() re-syncs .active from the widget's cached state, which doesn't
# know about merges/retirements done directly in code.
final_concepts = [
    {"name": c.name, "prompt": c.prompt}
    for c_id, c in l.concepts.items()
    if c_id not in RETIRED_CONCEPT_IDS
]

frozen = [{"id": f"L{i:02d}", "source": "lloom", "name": r["name"], "prompt": r["prompt"]}
          for i, r in enumerate(final_concepts)]
json.dump(frozen, open(os.path.join(OUTPUT_DIR, "frozen_concepts.json"), "w"), indent=2)
print(f"Froze {len(frozen)} concepts.")

Froze 8 concepts.


## Section 2.4:prints the concepts we came up with from these we only finalized 5 before

In [17]:
print(f"===== {l.name if hasattr(l, 'name') else 'l'} concepts ({len(l.concepts)} total) =====\n")
for c_id, c in l.concepts.items():
    print(f"[{c_id}]")
    print(f"  name: {c.name}")
    print(f"  prompt: {c.prompt}")
    print()

===== l concepts (9 total) =====

[9db03388-7d8a-495d-94af-817a976a7bac]
  name: Workplace Problem Guidance
  prompt: Determine whether the commenter advises the post author on handling workplace problems through prioritization, documentation, boundaries, escalation, or management support.

[04330b54-bb09-4973-b644-38076e61f2ba]
  name: Career Planning Advice
  prompt: Determine whether the commenter advises the post author about career direction, job searching, certifications, education, resumes, compensation, or changing workplaces.

[1e6986fb-b8ee-43be-a587-9c7eb6e945c6]
  name: Emotional Support
  prompt: Determine whether the commenter validates, reassures, empathizes with, or ,encourages the post author and their emotional well-being. Includes validates,reassures, encourages, or celebrates the post author in response to their experiences or emotions.

[7878b017-ead7-45fc-bf1b-457cc8639e8f]
  name: Critical Pushback
  prompt: Determine whether the commenter challenges, questions, 